# Génération de format de chargement de tarifs

Génération de deux fichiers parquet pour les tarifs et pour les liens tarif-pdc.

In [ ]:
from datetime import datetime
import sys
from pathlib import Path
import json
import pandas as pd
# remonte à la racine du projet OCPI
sys.path.append(str(Path().resolve().parents[0]))
from source.tariff import TariffObject

In [ ]:
def tariffs_from_OCPI(locations: list, tariffs: list, schema: dict) -> tuple[pd.DataFrame, pd.DataFrame]:
    original_id = []
    original_last_updated = []
    raw = []
    start= []
    end = []
    for tarif in tariffs["data"]:
        assert TariffObject.is_valid_json(schema, tarif, verbose=False)
        original_id.append(tarif["country_code"] + tarif["party_id"] + tarif["id"])
        original_last_updated.append(tarif["last_updated"])
        raw.append(tarif)
        start.append(max(datetime.fromisoformat(tarif["last_updated"]), datetime.fromisoformat(tarif.get("start_date_time", datetime.min))))
        end.append(datetime.fromisoformat(tarif["end_date_time"]) if tarif.get("end_date_time") else None)
    tariff = pd.DataFrame({
        "original_id": original_id,
        "original_last_updated": original_last_updated,
        "raw": raw,
        "start": start,
        "end": end})
    
    id_pdc_itinerance = []
    id_tariff = []
    for location in locations["data"]:
        for evse in location["evses"]:
            id_pdc_itinerance.append(evse["evse_id"].replace("*", ""))
            tariff_id = location["country_code"] + location["party_id"]
            tariff_id+= evse["connectors"][0]["tariff_ids"][0]
            if tariff_id not in original_id:
                print(f"Tariff ID {tariff_id} not found in original tariffs")
            id_tariff.append(tariff_id)
    tariffpdc = pd.DataFrame({
        "id_pdc_itinerance": id_pdc_itinerance,
        "id_tariff": id_tariff})
    return (tariff, tariffpdc)

## Tesla

In [ ]:
with open('../data/tesla/FR_Supercharging_locations.json', 'r', encoding='utf-8') as f:
    tesla_locations = json.load(f)
with open('../data/tesla/FR_Supercharging_NTSLA_tariffs.json', 'r', encoding='utf-8') as f:
    tesla_tariffs = json.load(f)
len(tesla_locations["data"]), len(tesla_tariffs["data"])
with open("../source/schema.json") as f:
    schema = json.load(f)
original_id = []
original_last_updated = []
raw = []
start= []
end = []
for tarif in tesla_tariffs["data"]:
    assert TariffObject.is_valid_json(schema, tarif, verbose=False)
    original_id.append(tarif["country_code"] + tarif["party_id"] + tarif["id"])
    original_last_updated.append(tarif["last_updated"])
    raw.append(tarif)
    start.append(max(datetime.fromisoformat(tarif["last_updated"]), datetime.fromisoformat(tarif.get("start_date_time", datetime.min))))
    end.append(datetime.fromisoformat(tarif["end_date_time"]) if tarif.get("end_date_time") else None)
tariff = pd.DataFrame({
    "original_id": original_id,
    "original_last_updated": original_last_updated,
    "raw": raw,
    "start": start,
    "end": end})
tariff.to_parquet("../data/tesla/qualicharge_tariff.parquet", index=False)
#tariff
id_pdc_itinerance = []
id_tariff = []
for location in tesla_locations["data"]:
    for evse in location["evses"]:
        id_pdc_itinerance.append(evse["evse_id"].replace("*", ""))
        tariff_id = location["country_code"] + location["party_id"]
        tariff_id+= evse["connectors"][0]["tariff_ids"][0]
        if tariff_id not in original_id:
            print(f"Tariff ID {tariff_id} not found in original tariffs")
        id_tariff.append(tariff_id)
tariffpdc = pd.DataFrame({
    "id_pdc_itinerance": id_pdc_itinerance,
    "id_tariff": id_tariff})
tariffpdc.to_parquet("../data/tesla/qualicharge_tariffpdc.parquet", index=False)
tariffpdc

Tariff ID NLTSL5fa1c37d-9aa9-4778-805b-8a1c1299765d5fa1c37d-9aa9-4778-805b-8a1c1299765d not found in original tariffs
Tariff ID NLTSL5fa1c37d-9aa9-4778-805b-8a1c1299765d5fa1c37d-9aa9-4778-805b-8a1c1299765d5fa1c37d-9aa9-4778-805b-8a1c1299765d not found in original tariffs
Tariff ID NLTSL5fa1c37d-9aa9-4778-805b-8a1c1299765d5fa1c37d-9aa9-4778-805b-8a1c1299765d5fa1c37d-9aa9-4778-805b-8a1c1299765d5fa1c37d-9aa9-4778-805b-8a1c1299765d not found in original tariffs
Tariff ID NLTSL80791b33-aedd-4d7d-a0b8-1d831c44de6880791b33-aedd-4d7d-a0b8-1d831c44de68 not found in original tariffs
Tariff ID NLTSL80791b33-aedd-4d7d-a0b8-1d831c44de6880791b33-aedd-4d7d-a0b8-1d831c44de6880791b33-aedd-4d7d-a0b8-1d831c44de68 not found in original tariffs
Tariff ID NLTSL80791b33-aedd-4d7d-a0b8-1d831c44de6880791b33-aedd-4d7d-a0b8-1d831c44de6880791b33-aedd-4d7d-a0b8-1d831c44de6880791b33-aedd-4d7d-a0b8-1d831c44de68 not found in original tariffs
Tariff ID NLTSL80791b33-aedd-4d7d-a0b8-1d831c44de6880791b33-aedd-4d7d-a0b8-1

,id_pdc_itinerance,id_tariff
0,FRTSLEA4QY2C,NLTSL5fa1c37d-9aa9-4778-805b-8a1c1299765d
1,FRTSLEA4T0JE,NLTSL5fa1c37d-9aa9-4778-805b-8a1c1299765d5fa1c...
2,FRTSLEA6K9GP,NLTSL5fa1c37d-9aa9-4778-805b-8a1c1299765d5fa1c...
3,FRTSLEA6LMC5,NLTSL5fa1c37d-9aa9-4778-805b-8a1c1299765d5fa1c...
4,FRTSLE0003DB,NLTSL80791b33-aedd-4d7d-a0b8-1d831c44de68
...,...,...
3893,FRTSLECM5I9O,NLTSL64395007-0c2e-45bf-a948-1148c3a1581564395...
3894,FRTSLECM5KYZ,NLTSL64395007-0c2e-45bf-a948-1148c3a1581564395...
3895,FRTSLECM5QVJ,NLTSL64395007-0c2e-45bf-a948-1148c3a1581564395...
3896,FRTSLECM5QVG,NLTSL64395007-0c2e-45bf-a948-1148c3a1581564395...


## Total

In [ ]:

with open('../data/total/Tarifs.json', 'r', encoding='utf-8') as f:
    total_tarifs = json.load(f)
len(total_tarifs)

65

## Driveco

In [ ]:

with open('../data/driveco/Tarifs.json', 'r', encoding='utf-8') as f:
    driveco_tarifs = json.load(f)
len(driveco_tarifs)

85